In [6]:
import os
import pandas as pd
from PIL import Image
from collections import Counter
from tabulate import tabulate # 아까 설치한 tabulate 활용

def summarize_recursive(data_path="/data3/jinsol/", max_depth=2):
    print(f"🔍 Scanning directory: {data_path}\n")
    
    summary = []
    
    # os.walk를 사용하여 하위 디렉토리까지 탐색
    for root, dirs, files in os.walk(data_path):
        # 현재 탐색 중인 폴더의 깊이(Depth) 계산
        depth = root.replace(data_path, "").count(os.sep)
        if depth >= max_depth: # 너무 깊이 들어가지 않도록 제한 (조절 가능)
            continue
            
        folder_name = os.path.basename(root) if root != data_path else "ROOT"
        num_dirs = len(dirs)
        num_files = len(files)
        
        # 확장자 분석 (예: .svs가 몇 개인지 등)
        ext_counter = Counter([os.path.splitext(f)[1].lower() for f in files])
        ext_summary = ", ".join([f"{ext}({count})" for ext, count in ext_counter.items()])
        
        # 해당 폴더의 총 용량 계산 (MB)
        total_size = sum(os.path.getsize(os.path.join(root, f)) for f in files) / (1024 * 1024)
        
        summary.append({
            "Depth": "  " * depth + "┗ " + folder_name,
            "Sub-Dirs": num_dirs,
            "Files": num_files,
            "Extensions": ext_summary if ext_summary else "Empty",
            "Total Size(MB)": f"{total_size:.2f}"
        })

    # 표로 출력
    df = pd.DataFrame(summary)
    print("--- Directory Structure & File Summary ---")
    print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))

    # [추가] 만약 특정 파일(예: TCGA 내부 파일)의 세부 정보를 보고 싶다면?
    # 아래 코드를 활성화해서 샘플링 확인 가능
    print("\n--- Top 5 Files Sample ---")
    file_samples = []
    for root, _, files in os.walk(data_path):
        for f in files[:5]: # 각 폴더당 최대 5개만 샘플링
            file_samples.append({"Location": root, "File": f})
        if len(file_samples) > 10: break # 전체 샘플 10개 제한
    
    sample_df = pd.DataFrame(file_samples)
    print(tabulate(sample_df, headers='keys', tablefmt='simple', showindex=False))

if __name__ == "__main__":
    target_path = "/data3/jinsol/"
    if os.path.exists(target_path):
        # max_depth=2는 ROOT -> TCGA -> 그 다음 레벨까지만 보여줍니다.
        summarize_recursive(target_path, max_depth=3)
    else:
        print(f"Path not found: {target_path}")

🔍 Scanning directory: /data3/jinsol/

--- Directory Structure & File Summary ---
+----------------------+------------+---------+--------------+------------------+
| Depth                |   Sub-Dirs |   Files | Extensions   |   Total Size(MB) |
|----------------------+------------+---------+--------------+------------------|
| ┗ ROOT               |          1 |       0 | Empty        |             0    |
| ┗ TCGA               |          2 |       0 | Empty        |             0    |
| ┗ clinical           |          0 |      17 | .tsv(17)     |            36.68 |
| ┗ features_conch_256 |          2 |       0 | Empty        |             0    |
| ┗ pt_files           |         17 |       0 | Empty        |             0    |
| ┗ h5_files           |         17 |       0 | Empty        |             0    |
+----------------------+------------+---------+--------------+------------------+

--- Top 5 Files Sample ---
Location                                                  File
--------

In [1]:
import torch
import h5py
import pandas as pd
import os

def analyze_data_types(base_path="/data3/jinsol/TCGA/"):
    print("=== 1. Clinical Data (.tsv) Sample ===")
    clinical_path = os.path.join(base_path, "clinical")
    if os.path.exists(clinical_path):
        tsv_files = [f for f in os.listdir(clinical_path) if f.endswith('.tsv')]
        if tsv_files:
            df = pd.read_csv(os.path.join(clinical_path, tsv_files[0]), sep='\t')
            print(f"File: {tsv_files[0]}")
            print(f"Columns: {df.columns.tolist()[:10]} ...") # 상위 10개 컬럼만
            print(f"Shape: {df.shape}")

    print("\n=== 2. Feature Data (.pt) Shape Check ===")
    pt_root = os.path.join(base_path, "features_conch_256/pt_files")
    # 첫 번째 암종 폴더의 첫 번째 pt 파일을 찾음
    found_pt = False
    for root, dirs, files in os.walk(pt_root):
        for f in files:
            if f.endswith('.pt'):
                pt_data = torch.load(os.path.join(root, f), map_location='cpu')
                print(f"Sample PT File: {f}")
                print(f"Data Type: {type(pt_data)}")
                if torch.is_tensor(pt_data):
                    print(f"Tensor Shape: {pt_data.shape} (N_patches, Embedding_dim)")
                found_pt = True
                break
        if found_pt: break

    print("\n=== 3. H5 Data (.h5) Structure Check ===")
    h5_root = os.path.join(base_path, "features_conch_256/h5_files")
    found_h5 = False
    for root, dirs, files in os.walk(h5_root):
        for f in files:
            if f.endswith('.h5'):
                with h5py.File(os.path.join(root, f), 'r') as hf:
                    print(f"Sample H5 File: {f}")
                    print(f"Keys in H5: {list(hf.keys())}")
                    if 'coords' in hf:
                        print(f"Coords Shape: {hf['coords'].shape}")
                found_h5 = True
                break
        if found_h5: break

if __name__ == "__main__":
    analyze_data_types()

=== 1. Clinical Data (.tsv) Sample ===
File: TCGA-KICH-clinical.tsv
Columns: ['project.project_id', 'cases.case_id', 'cases.consent_type', 'cases.days_to_consent', 'cases.days_to_lost_to_followup', 'cases.disease_type', 'cases.index_date', 'cases.lost_to_followup', 'cases.primary_site', 'cases.submitter_id'] ...
Shape: (385, 210)

=== 2. Feature Data (.pt) Shape Check ===
Sample PT File: TCGA-A7-A0CH-01Z-00-DX1.48ADDD0B-527A-404C-B794-7D7D35BF861F.pt
Data Type: <class 'torch.Tensor'>
Tensor Shape: torch.Size([18703, 512]) (N_patches, Embedding_dim)

=== 3. H5 Data (.h5) Structure Check ===
Sample H5 File: TCGA-AR-A0TQ-01A-01-BSA.ed7f9356-b8cc-4e4b-8caa-d977e4441831.h5
Keys in H5: ['coords', 'features']
Coords Shape: (13890, 2)


In [4]:
import pandas as pd
import torch
import h5py
import os
import glob

# 설정: 파일 경로 (사용자 환경에 맞춰 수정 가능)
base_path = "/data3/jinsol/TCGA"
clinical_dir = os.path.join(base_path, "clinical")
pt_dir = os.path.join(base_path, "features_conch_256/pt_files")
h5_dir = os.path.join(base_path, "features_conch_256/h5_files")

def summarize_data():
    print("=== [1] Clinical Data (라벨 데이터 후보) 요약 ===")
    clinical_files = glob.glob(os.path.join(clinical_dir, "*.tsv"))
    if clinical_files:
        # 대표로 첫 번째 파일 분석
        df = pd.read_csv(clinical_files[0], sep='\t')
        
        print(f"파일명: {os.path.basename(clinical_files[0])}")
        print(f"전체 컬럼 수: {len(df.columns)}")
        
        # 라벨로 주로 쓰이는 주요 컬럼 예시 (TCGA 표준 명칭 기준)
        target_candidates = [
            'cases.disease_type',          # 암 종류 (분류)
            'cases.samples.0.sample_type', # 샘플 타입 (정상 vs 종양)
            'cases.diagnoses.0.tumor_stage', # 암 병기 (Stage I, II...)
            'cases.diagnoses.0.vital_status', # 생존 여부 (Alive/Dead)
            'cases.diagnoses.0.days_to_death' # 생존 기간 (회귀/생존분석)
        ]
        
        available_targets = [c for c in target_candidates if c in df.columns]
        print("\n[라벨 활용 가능 데이터 예시]")
        for col in available_targets:
            unique_vals = df[col].dropna().unique()[:5]
            print(f"- {col}: {unique_vals} ...")
            
    print("\n" + "="*50)
    print("=== [2] Feature Data (.pt 파일) 구조 요약 ===")
    # 폴더 내의 첫 번째 .pt 파일 탐색
    pt_files = glob.glob(os.path.join(pt_dir, "*/*.pt"))
    if pt_files:
        sample_pt = torch.load(pt_files[0])
        print(f"파일명: {os.path.basename(pt_files[0])}")
        print(f"피쳐 벡터 형태 (Shape): {sample_pt.shape}")
        print(f" - 패치 개수: {sample_pt.shape[0]}")
        print(f" - 임베딩 차원: {sample_pt.shape[1]}")
        print(f"피쳐 값 예시 (첫 번째 패치의 앞 5개 값):")
        print(f" {sample_pt[0, :5].tolist()}...")

    print("\n" + "="*50)
    print("=== [3] H5 Data (좌표 및 피쳐 정보) 요약 ===")
    h5_files = glob.glob(os.path.join(h5_dir, "*/*.h5"))
    if h5_files:
        if h5_files:
            with h5py.File(h5_files[0], 'r') as h5_py_file:
                print(f"파일명: {os.path.basename(h5_files[0])}")
                print(f"H5 내부 키 목록: {list(h5_py_file.keys())}")
            
                if 'coords' in h5_py_file:
                    coords = h5_py_file['coords'][:]
                    print(f"좌표 데이터 형태: {coords.shape} (N_patches, [x, y])")
                    print(f"좌표 값 예시: {coords[0].tolist()}")

if __name__ == "__main__":
    summarize_data()

=== [1] Clinical Data (라벨 데이터 후보) 요약 ===
파일명: TCGA-KICH-clinical.tsv
전체 컬럼 수: 210

[라벨 활용 가능 데이터 예시]
- cases.disease_type: ['Adenomas and Adenocarcinomas'] ...

=== [2] Feature Data (.pt 파일) 구조 요약 ===
파일명: TCGA-A7-A0CH-01Z-00-DX1.48ADDD0B-527A-404C-B794-7D7D35BF861F.pt
피쳐 벡터 형태 (Shape): torch.Size([18703, 512])
 - 패치 개수: 18703
 - 임베딩 차원: 512
피쳐 값 예시 (첫 번째 패치의 앞 5개 값):
 [0.8941792845726013, -0.4052782952785492, 0.4849233031272888, 0.32258275151252747, -0.03175115957856178]...

=== [3] H5 Data (좌표 및 피쳐 정보) 요약 ===
파일명: TCGA-AR-A0TQ-01A-01-BSA.ed7f9356-b8cc-4e4b-8caa-d977e4441831.h5
H5 내부 키 목록: ['coords', 'features']
좌표 데이터 형태: (13890, 2) (N_patches, [x, y])
좌표 값 예시: [71651, 20518]


In [5]:
import pandas as pd
import glob
import os

# 설정: clinical 파일들이 모여있는 경로
clinical_dir = "/data3/jinsol/TCGA/clinical"

def analyze_labels(directory):
    tsv_files = glob.glob(os.path.join(directory, "*.tsv"))
    
    all_data = []
    for f in tsv_files:
        # 필요한 컬럼만 선택해서 읽기 (메모리 절약)
        # 보통 project_id가 암종을 가장 명확하게 구분함 (예: TCGA-BRCA)
        df = pd.read_csv(f, sep='\t', usecols=['project.project_id', 'cases.disease_type'])
        all_data.append(df)
    
    # 모든 파일 합치기
    combined_df = pd.concat(all_data, ignore_index=True)
    
    print("=== [라벨 분석 결과] ===")
    
    # 1. Project ID 기준 (가장 권장되는 라벨)
    print(f"\n1. 'project.project_id' 기준 클래스 (총 {combined_df['project.project_id'].nunique()}개):")
    print(combined_df['project.project_id'].value_counts())
    
    print("\n" + "-"*30)
    
    # 2. Disease Type 기준 (더 상세한 명칭)
    print(f"2. 'cases.disease_type' 기준 클래스 (총 {combined_df['cases.disease_type'].nunique()}개):")
    print(combined_df['cases.disease_type'].value_counts().head(10)) # 상위 10개만 출력

if __name__ == "__main__":
    analyze_labels(clinical_dir)

=== [라벨 분석 결과] ===

1. 'project.project_id' 기준 클래스 (총 17개):
project.project_id
TCGA-BRCA    5546
TCGA-OV      4223
TCGA-LGG     3721
TCGA-GBM     3495
TCGA-LUAD    2466
TCGA-BLCA    2412
TCGA-LUSC    1893
TCGA-COAD    1801
TCGA-PRAD    1243
TCGA-TGCT    1013
TCGA-MESO     609
TCGA-ACC      568
TCGA-PCPG     551
TCGA-UCS      444
TCGA-KICH     385
TCGA-DLBC     382
TCGA-CHOL     378
Name: count, dtype: int64

------------------------------
2. 'cases.disease_type' 기준 클래스 (총 18개):
cases.disease_type
Gliomas                                        7205
Adenomas and Adenocarcinomas                   5918
Ductal and Lobular Neoplasms                   5394
Cystic, Mucinous and Serous Neoplasms          4613
Transitional Cell Papillomas and Carcinomas    2400
Squamous Cell Neoplasms                        1900
Germ Cell Neoplasms                            1013
Mesothelial Neoplasms                           609
Paragangliomas and Glomus Tumors                551
Acinar Cell Neoplasms         

In [1]:
import os

def print_directory_tree(startpath):
    print(f"Directory structure for: {startpath}")
    for root, dirs, files in os.walk(startpath):
        # 현재 경로가 시작 경로에서 얼마나 깊은지 계산
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        
        # 파일까지 보고 싶다면 아래 주석을 해제하세요
        # subindent = ' ' * 4 * (level + 1)
        # for f in files:
        #     print(f"{subindent}{f}")

# 경로 설정
target_path = '/data3/jinsol/'

if os.path.exists(target_path):
    print_directory_tree(target_path)
else:
    print("해당 경로를 찾을 수 없습니다.")

Directory structure for: /data3/jinsol/
/
TCGA/
    clinical/
    features_conch_256/
        pt_files/
            TCGA-BRCA/
            TCGA-LUSC/
            TCGA-BLCA/
            TCGA-PRAD/
            TCGA-COAD/
            TCGA-KICH/
            TCGA-ACC/
            TCGA-MESO/
            TCGA-OV/
            TCGA-GBM/
            TCGA-PCPG/
            TCGA-TGCT/
            TCGA-DLBC/
            TCGA-CHOL/
            TCGA-UCS/
            TCGA-LUAD/
            TCGA-LGG/
        h5_files/
            TCGA-BRCA/
            TCGA-LUSC/
            TCGA-BLCA/
            TCGA-PRAD/
            TCGA-COAD/
            TCGA-KICH/
            TCGA-ACC/
            TCGA-MESO/
            TCGA-OV/
            TCGA-GBM/
            TCGA-PCPG/
            TCGA-TGCT/
            TCGA-DLBC/
            TCGA-CHOL/
            TCGA-UCS/
            TCGA-LUAD/
            TCGA-LGG/


In [1]:
import pandas as pd

# tsv 파일 경로 지정 (BRCA 파일 예시)
tsv_path = '/data3/jinsol/TCGA/clinical/TCGA-BRCA-clinical.tsv'

# pandas로 tsv 파일 읽기 (구분자를 탭 '\t'으로 설정)
df = pd.read_csv(tsv_path, sep='\t')

# 1. 어떤 컬럼(데이터 종류)들이 있는지 확인
print("📊 컬럼 목록:", df.columns.tolist())
print("-" * 50)

# 2. 위에서부터 5줄만 표 형태로 깔끔하게 출력해서 내용물 확인
display(df.head())

📊 컬럼 목록: ['project.project_id', 'cases.case_id', 'cases.consent_type', 'cases.days_to_consent', 'cases.days_to_lost_to_followup', 'cases.disease_type', 'cases.index_date', 'cases.lost_to_followup', 'cases.primary_site', 'cases.submitter_id', 'demographic.age_at_index', 'demographic.age_is_obfuscated', 'demographic.cause_of_death', 'demographic.cause_of_death_source', 'demographic.country_of_birth', 'demographic.country_of_residence_at_enrollment', 'demographic.days_to_birth', 'demographic.days_to_death', 'demographic.demographic_id', 'demographic.education_level', 'demographic.ethnicity', 'demographic.gender', 'demographic.marital_status', 'demographic.occupation_duration_years', 'demographic.population_group', 'demographic.premature_at_birth', 'demographic.race', 'demographic.submitter_id', 'demographic.vital_status', 'demographic.weeks_gestation_at_birth', 'demographic.year_of_birth', 'demographic.year_of_death', 'diagnoses.adrenal_hormone', 'diagnoses.age_at_diagnosis', 'diagnoses.a

/tmp/ipykernel_8994/699787509.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep='\t')


,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-E2-A1IU,...,'--,'--,'--,'--,1b884f21-eb24-467f-aba2-208af17070b9,Adjuvant,no,'--,'--,"Radiation Therapy, NOS"
1,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-E2-A1IU,...,'--,'--,'--,'--,27868bc3-23c8-5e85-a0e2-314e6cdf9b2a,Adjuvant,yes,Treatment Ongoing,'--,Hormone Therapy
2,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-E2-A1IU,...,'--,'--,'--,'--,aedf144c-6b7b-4d76-a3cb-4271aef10f1d,First-Line Therapy,yes,'--,'--,"Surgery, NOS"
3,TCGA-BRCA,0045349c-69d9-4306-a403-c9c1fa836644,Informed Consent,76,'--,Adenomas and Adenocarcinomas,Diagnosis,'--,Breast,TCGA-A1-A0SB,...,'--,'--,'--,'--,0a534cae-de91-5e77-a3e7-b52d46bd3966,First-Line Therapy,yes,'--,'--,"Surgery, NOS"
4,TCGA-BRCA,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,Informed Consent,19,'--,Adnexal and Skin Appendage Neoplasms,Diagnosis,No,Breast,TCGA-A2-A04W,...,'--,'--,'--,'--,024faa94-ec57-4d14-b919-62dcab409958,Adjuvant,yes,Treatment Ongoing,'--,Bisphosphonate Therapy


In [2]:
import pandas as pd

# tsv 파일 경로 지정 (BRCA 파일 예시)
tsv_path = '/data3/jinsol/TCGA/clinical/TCGA-BRCA-clinical.tsv'

# pandas로 tsv 파일 읽기
df = pd.read_csv(tsv_path, sep='\t', low_memory=False)

print("=== 1. project.project_id (암종 라벨) ===")
# 어떤 암종 프로젝트 이름이 몇 개 있는지 출력
print(df['project.project_id'].value_counts())
print("\n" + "=" * 50 + "\n")

print("=== 2. cases.submitter_id (환자 ID) ===")
# 전체 행 개수와 고유한 환자 수 비교
print(f"전체 데이터 줄(Row) 수: {len(df)}개")
print(f"고유한 환자(Patient) 수: {df['cases.submitter_id'].nunique()}명\n")

# 환자별로 데이터가 몇 개씩 있는지 확인 (가장 많은 상위 10명만 출력)
print("[환자 ID별 데이터 개수 (Top 10)]")
print(df['cases.submitter_id'].value_counts().head(10))

=== 1. project.project_id (암종 라벨) ===
project.project_id
TCGA-BRCA    5546
Name: count, dtype: int64


=== 2. cases.submitter_id (환자 ID) ===
전체 데이터 줄(Row) 수: 5546개
고유한 환자(Patient) 수: 1098명

[환자 ID별 데이터 개수 (Top 10)]
cases.submitter_id
TCGA-GM-A2DA    40
TCGA-A2-A04P    33
TCGA-A2-A3XU    22
TCGA-A2-A3XY    21
TCGA-A2-A0EW    20
TCGA-E9-A2JS    19
TCGA-AO-A03N    19
TCGA-A2-A04Y    17
TCGA-AO-A0J9    17
TCGA-A2-A04V    17
Name: count, dtype: int64
